# 13U Team Manager Dashboard (OBA Rules)

**Rule Authority:** Baseball Canada Section 4.4 as adopted by Baseball Ontario (OBA)

| Setting | Value |
|---|---|
| Bases | 75 ft |
| Mound | 50 ft |
| Innings | 7 |
| Max pitches/day | 85 |
| Mercy | 18 after 3, 15 after 4, 10 after 5 |

Three modules:
1. **Practice Architect** — minute-by-minute plans with rep tracking
2. **Lineup & Position Tracker** — developmental rotation with exposure algorithm
3. **Pitch Count & Recovery** — OBA-compliant rest tracking

In [ ]:
# Load everything
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from dashboard import *

---
## Module 1: Practice Architect

Change the inputs below and re-run to generate a new plan.

In [ ]:
# ── PRACTICE INPUTS ──
DURATION = 90          # minutes (90 or 120)
PLAYERS = 12           # players at practice
COACHES = 3            # coaches / volunteers

# Focus options:
#   "Middle Infield Double Play Feeds and Turns"
#   "Cut-offs"
#   "1st & 3rd Defense"
#   "Situational Baserunning"
#   "General"
FOCUS = "Middle Infield Double Play Feeds and Turns"

result = generate_practice(DURATION, PLAYERS, COACHES, FOCUS)
print_practice(result)

---
## Module 2: Developmental Lineup & Position Tracker

### Step 1: Initialize your roster (run once at the start of the season)

In [ ]:
# ── ROSTER (edit names to match your team) ──
MY_ROSTER = [
    "Marcus", "Devon", "Jaylen",
    "Tyler", "Noah", "Ethan",
    "Carlos", "Aiden", "Bryce",
    "Liam", "Mason", "Kai",
]

df = initialize_roster(MY_ROSTER)
print(f"Roster initialized: {len(MY_ROSTER)} players")
df

### Step 2: Load existing innings (skip if starting fresh)

Edit the lines below to match your team's current season totals. Format: `df.loc[df["Player"] == "Name", ["POS1", "POS2"]] = [innings1, innings2]`

In [ ]:
# ── EXAMPLE: Pre-load season innings ──
# Delete or edit these to match your actual team data.

df.loc[df["Player"] == "Marcus", ["SS", "P"]] = [12, 10]
df.loc[df["Player"] == "Devon", ["SS", "P"]] = [11, 8]
df.loc[df["Player"] == "Jaylen", ["P", "SS"]] = [10, 11]
df.loc[df["Player"] == "Tyler", ["C"]] = 14
df.loc[df["Player"] == "Noah", ["1B"]] = 10
df.loc[df["Player"] == "Ethan", ["2B", "3B"]] = [8, 6]
df.loc[df["Player"] == "Carlos", ["LF", "CF"]] = [7, 5]
df.loc[df["Player"] == "Aiden", ["RF", "LF"]] = [9, 4]
df.loc[df["Player"] == "Bryce", ["3B", "1B"]] = [7, 6]
df.loc[df["Player"] == "Liam", ["CF", "RF"]] = [6, 8]
df.loc[df["Player"] == "Mason", ["2B", "SS"]] = [5, 3]
df.loc[df["Player"] == "Kai", ["LF", "RF"]] = [4, 3]

df.loc[df["Player"].isin(["Marcus", "Devon", "Jaylen", "Tyler"]), "Games"] = 6

_save_roster(df)
print("Season data loaded.")
df

### Step 3: Generate a lineup

- `locked` — positions you don't want the algorithm to touch (e.g., your only catcher)
- `force_outfield` — players who must play OF this game
- `designated_pitchers` — players who will pitch (OBA rule: they cannot catch same day)

In [ ]:
# ── LINEUP INPUTS ──
lineup = suggest_lineup(
    innings=7,
    locked={"Tyler": "C"},
    force_outfield=["Marcus", "Devon", "Jaylen"],
    designated_pitchers=["Marcus", "Devon"],
)
print_lineup(lineup)

### Step 4: Update stats after a game

Run this after each game to keep the exposure algorithm accurate.

In [ ]:
# ── POST-GAME UPDATE (edit after each game) ──
# Format: {"PlayerName": {"POS": innings_at_that_position, ...}}

game_result = {
    "Marcus": {"LF": 4, "CF": 3},
    "Devon": {"RF": 3, "CF": 4},
    "Jaylen": {"LF": 3, "RF": 4},
    "Tyler": {"C": 7},
    "Noah": {"1B": 4, "P": 3},
    "Ethan": {"SS": 4, "2B": 3},
    "Carlos": {"3B": 4, "LF": 3},
    "Aiden": {"P": 4, "1B": 3},
    "Bryce": {"2B": 4, "3B": 3},
    "Liam": {"CF": 4, "RF": 3},
    "Mason": {"SS": 3, "3B": 4},
    "Kai": {"RF": 3, "2B": 4},
}

# Players who sat an inning (list names)
sat = []

updated = update_season_stats(game_result, sat_players=sat)
print("Stats updated.")
updated

---
## Module 3: Pitch Count & Recovery Monitor

### Log pitches after a game

In [ ]:
# ── LOG PITCHES (run after each game for every pitcher who threw) ──
# Automatically checks OBA limits and flags violations.

log_pitches("Marcus", 72, "2026-05-12")
log_pitches("Devon", 45, "2026-05-13")
log_pitches("Jaylen", 20, "2026-05-11")
log_pitches("Tyler", 82, "2026-05-10")

### Check pitcher availability

Change the date to your next game/practice date.

In [ ]:
# ── PITCHER STATUS (change date to your next game) ──
CHECK_DATE = "2026-05-14"

status = pitcher_status(CHECK_DATE)
print_pitcher_status(status)

---
## Quick Reference: OBA 13U Pitch Count Rules

| Pitches | Rest Required |
|---|---|
| 1-30 | None |
| 31-45 | 1 calendar day |
| 46-60 | 2 calendar days |
| 61-75 | 3 calendar days |
| 76-85 | 4 calendar days |

**Daily max:** 85 | **2-day max:** 85 | **4-day max:** 120

**Consecutive days:** Max 3, only if first 2 days combined ≤ 30 pitches. 4 consecutive days prohibited.

**Pitcher/Catcher:** A player who pitches cannot catch the remainder of that calendar day (Section 4.4(j)).